# Titanic (1997): HTML and API

Read shot data from Cinemetrics and film details from Wikidata. Check the IMDb IDs, then combine the records.

Start Jupyter in the project root and run from top to bottom. Saved files are used by default; set `live = True` for new requests.

`Q44578` is the Wikidata item for this film. The numbers are an ID, not a year or score.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import time
import requests
from bs4 import BeautifulSoup

folder = Path.cwd()
if not (folder / "01_raw").is_dir():
    raise RuntimeError("Start Jupyter from the project root, beside the 01_raw folder.")
live = False
html_url = "https://cinemetrics.uchicago.edu/movie/0e9a80c6-ce7f-45b3-976b-6211e51bd19b"
api_url = "https://www.wikidata.org/w/rest.php/wikibase/v1/entities/items/Q44578"
headers = {"User-Agent": "ADS206-CW1-film-demo/0.1 (educational, read-only)"}
data_dir = folder / "01_raw/single_film_demo/titanic_1997"
last_run = data_dir / "latest_run.txt"
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") if live else last_run.read_text().strip()
run_dir = data_dir / run_id
run_dir.mkdir(parents=True, exist_ok=True)
print("Mode:", "live" if live else "saved files")


Mode: saved files


### Request and save

This function downloads a response and saves the original file. It also records the URL, time and status. No automatic retries are included.

In [2]:
def fetch(url, name):
    if not live:
        return (run_dir / name).read_bytes()
    started = datetime.now(timezone.utc).isoformat()
    clock = time.perf_counter()
    response = requests.get(url, headers=headers, timeout=30)
    (run_dir / name).write_bytes(response.content)
    log = {"time": started, "url": url, "status": response.status_code,
           "seconds": round(time.perf_counter() - clock, 3), "retry_count": 0,
           "file": name, "collector": "AI-assisted demo run"}
    with (run_dir / "requests.jsonl").open("a", encoding="utf-8") as file:
        file.write(json.dumps(log) + "\n")
    print(name, "status:", response.status_code)
    response.raise_for_status()
    time.sleep(1.1)
    return response.content


## 1. HTML: shot data

Fetch the page. Read the title, IMDb link and overall shot statistics. The lower table contains separate section statistics.

In [3]:
soup = BeautifulSoup(fetch(html_url, "cinemetrics.html"), "html.parser")
heading = soup.find("header").get_text(" ", strip=True)
title = re.search(r"^(.*?)\s*\(\s*(\d{4})", heading)
assert title, "Check the film heading."
imdb_ids = {re.search(r"tt\d+", link["href"]).group()
            for link in soup.find_all("a", href=re.compile(r"imdb\.com/title/tt\d+"))}
assert len(imdb_ids) == 1, "Check the IMDb links."
stats = {}
for button in soup.find_all("button"):
    parts = button.find_all("span", recursive=False)
    if len(parts) == 2:
        label = parts[0].get_text(strip=True).rstrip(":")
        if label in {"NoS", "LEN", "ASL", "MSL"}:
            assert label not in stats, "Duplicate field: " + label
            stats[label] = parts[1].get_text(strip=True)
assert set(stats) == {"NoS", "LEN", "ASL", "MSL"}, "Check the shot fields."
html_data = {"title_html": title[1].strip(), "year_page": int(title[2]),
             "imdb_id_html": next(iter(imdb_ids)),
             "shot_count": int(stats["NoS"].replace(",", "")),
             "measured_duration": stats["LEN"],
             "asl_seconds": float(stats["ASL"]), "msl_seconds": float(stats["MSL"])}
print(json.dumps(html_data, indent=2))


{
  "title_html": "Titanic",
  "year_page": 1997,
  "imdb_id_html": "tt0120338",
  "shot_count": 2673,
  "measured_duration": "03:06:45.6",
  "asl_seconds": 4.2,
  "msl_seconds": 3.0
}


## 2. API: film details

Read the REST API response as JSON. `P345` holds IMDb IDs; `P577` holds release dates. Keep the available release years.

In [4]:
item = json.loads(fetch(api_url, "wikidata.json"))
assert item["id"] == "Q44578", "Check the API item."

def values(prop):
    result = []
    for row in item["statements"].get(prop, []):
        value = row.get("value", {})
        if row["rank"] != "deprecated" and value.get("type") == "value":
            result.append(value["content"])
    return result

imdb_ids = set(values("P345"))
assert len(imdb_ids) == 1, "Check the API IMDb ID."
years = sorted({int(date["time"][1:5]) for date in values("P577")
                if date.get("precision", 0) >= 9 and re.match(r"^\+\d{4}-", date["time"])})
api_data = {"wikidata_id": item["id"], "title_api": item["labels"]["en"],
            "description_api": item["descriptions"].get("en", ""),
            "imdb_id_api": next(iter(imdb_ids)), "release_years_api": years}
print(json.dumps(api_data, indent=2))


{
  "wikidata_id": "Q44578",
  "title_api": "Titanic",
  "description_api": "1997 film directed by James Cameron",
  "imdb_id_api": "tt0120338",
  "release_years_api": [
    1997,
    1998
  ]
}


## 3. Match and combine

Check that both sources refer to the same film, then save one record. The measured edition and completeness still need checking.

In [5]:
if "html_data" not in globals() or "api_data" not in globals():
    raise RuntimeError("Run the cells above first.")
assert html_data["imdb_id_html"] == api_data["imdb_id_api"], "IMDb IDs do not match."
assert html_data["title_html"].casefold() == api_data["title_api"].casefold(), "Check the titles."
assert html_data["year_page"] in api_data["release_years_api"], "Check the years."
film = {**html_data, **api_data, "html_source_url": html_url, "api_source_url": api_url,
        "measurement_scope_status": "pending version, speed and boundary checks",
        "selected_for_main_analysis": False, "evidence_run": run_id}
if live:
    (folder / "02_audit/single_film_demo.json").write_text(json.dumps(film, indent=2), encoding="utf-8")
    last_run.write_text(run_id, encoding="utf-8")
print("IMDb match:", film["imdb_id_api"])
print(json.dumps(film, indent=2))


IMDb match: tt0120338
{
  "title_html": "Titanic",
  "year_page": 1997,
  "imdb_id_html": "tt0120338",
  "shot_count": 2673,
  "measured_duration": "03:06:45.6",
  "asl_seconds": 4.2,
  "msl_seconds": 3.0,
  "wikidata_id": "Q44578",
  "title_api": "Titanic",
  "description_api": "1997 film directed by James Cameron",
  "imdb_id_api": "tt0120338",
  "release_years_api": [
    1997,
    1998
  ],
  "html_source_url": "https://cinemetrics.uchicago.edu/movie/0e9a80c6-ce7f-45b3-976b-6211e51bd19b",
  "api_source_url": "https://www.wikidata.org/w/rest.php/wikibase/v1/entities/items/Q44578",
  "measurement_scope_status": "pending version, speed and boundary checks",
  "selected_for_main_analysis": false,
  "evidence_run": "20261107T162303612368Z"
}


Sources: Cinemetrics and Wikidata (URLs above). Methods follow the Week 4 lecture: requests, HTML parsing and JSON parsing.

[Wikidata REST API](https://www.wikidata.org/wiki/Wikidata:REST_API)
